# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [18]:
# imports
import os
from IPython.display import Markdown, display, update_display
from dotenv import load_dotenv
from openai import OpenAI, base_url

from week1.day1 import user_prompt

In [6]:
# constants
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2:1b'

# set up environment
load_dotenv(override=True)

# initialize OpenAI client
api_key = os.getenv('OPENAI_API_KEY')
openai = OpenAI(api_key=api_key)

# initialize Ollama client
OLLAMA_BASE_URL = "http://localhost:11434/v1/"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [24]:
# here is the question; type over this to ask something new
system_prompt = """
You are a tutor that teaches programming. Give first a short summary and then explain every part of the code.
"""

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

user_prompt = """
Please explain what this code does and why:
"""

In [28]:
# Get gpt-4o-mini to answer, with streaming
def stream_gpt_response(question):
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [30]:
# Get Llama 3.2 to answer
def ollama_response(question):
    response = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
    )
    result = response.choices[0].message.content
    return result

In [36]:
# Get streaming answer for multiple models
def stream_response(question, client, model):
    stream = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [38]:
code_snippet = """
def stream_gpt_response(question):
    from IPython.display import Markdown, display, update_display

    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)
"""

#stream_gpt_response(question=f"{user_prompt}\n{code_snippet}")
# print(ollama_response(question=f"{user_prompt}\n{code_snippet}"))
stream_response(question=f"{user_prompt}\n{code_snippet}", client=openai, model=MODEL_GPT)

This code defines a function `stream_gpt_response` that sends user questions to the OpenAI GPT model (using a streaming method) and displays the responses in a Markdown format, updating the display as new parts of the response are generated. The function utilizes the `IPython` display module to facilitate real-time updates of the output in an interactive notebook environment.

Here's a breakdown of each part of the code:

### Code Breakdown

1. **Function Definition**:
   ```python
   def stream_gpt_response(question):
   ```
   This line defines a new function named `stream_gpt_response` that takes a single argument `question`, which is expected to be a string representing the user's query to the GPT model.

2. **Importing Required Libraries**:
   ```python
   from IPython.display import Markdown, display, update_display
   ```
   This line imports specific functions and classes (`Markdown`, `display`, `update_display`) from the `IPython.display` module. These are used for rendering Markdown text and updating the display in a Jupyter notebook environment.

3. **Creating a Streaming Chat Completion**:
   ```python
   stream = openai.chat.completions.create(
       model=MODEL_GPT,
       messages=[
           {"role": "system", "content": system_prompt},
           {"role": "user", "content": question}
       ],
       stream=True
   )
   ```
   - **`openai.chat.completions.create`**: This method from OpenAI's API is used to generate chat completions.
   - **`model=MODEL_GPT`**: Indicates the specific model being used (presumably defined elsewhere in the code).
   - **`messages`**: This argument is a list that contains the messages being sent to the model. It consists of:
     - A system message outlined by `system_prompt`, which could define the behavior or characteristics of the assistant (you need to ensure `system_prompt` is defined elsewhere).
     - A user message derived from the `question` provided to the function.
   - **`stream=True`**: This indicates that the completion should be streamed, allowing the response to be received in smaller chunks.

4. **Initializing the Response String**:
   ```python
   response = ""
   ```
   This initializes an empty string `response` to accumulate parts of the model's responses as they are streamed.

5. **Setting Up the Display Handle**:
   ```python
   display_handle = display(Markdown(""), display_id=True)
   ```
   This creates an initial display output that is empty (as markdown), and `display_id=True` allows this output to be updated later. The handle to this display is stored in `display_handle`.

6. **Iterating Over the Streamed Response**:
   ```python
   for chunk in stream:
   ```
   This line begins a loop that iterates over each chunk of data received from the streaming response from GPT. Each chunk corresponds to a part of the model's response.

7. **Updating the Response**:
   ```python
   response += chunk.choices[0].delta.content or ''
   ```
   Inside the loop, the response is updated by appending the content of each chunk. `chunk.choices[0].delta.content` accesses the content returned in the first choice of the chunk. The use of `or ''` ensures that if `content` is `None`, an empty string is added instead.

8. **Updating the Display**:
   ```python
   update_display(Markdown(response), display_id=display_handle.display_id)
   ```
   Finally, the `update_display` function is called to update the Markdown display in the notebook with the current state of the `response`. The `display_id` is used to identify which display to update, ensuring it refreshes the same area rather than creating new output cells.

### Summary
Overall, this code functions as a real-time stream handler for responses generated by the OpenAI GPT model, displaying each part of the response interactively within a Jupyter notebook environment. It's particularly useful for scenarios where responses might be lengthy and you want users to see them as they are being generated, rather than waiting for the entire completion to finish before displaying anything.